# Prospect v7.5.5 Normal-class Notebook — GPT-J-6B fp32

Drive-direct prospect workflow for `meta-llama/GPT-J-6B fp32`.


In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import importlib.metadata
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    'nnsight',
    'transformers',
    'accelerate',
    'scipy',
    'pandas',
    'wordfreq',
    'rapidfuzz',
    'huggingface_hub',
])

print('nnsight:', importlib.metadata.version('nnsight'))
print('transformers:', importlib.metadata.version('transformers'))
print('accelerate:', importlib.metadata.version('accelerate'))
print('scipy:', importlib.metadata.version('scipy'))
print('pandas:', importlib.metadata.version('pandas'))
print('wordfreq:', importlib.metadata.version('wordfreq'))
print('rapidfuzz:', importlib.metadata.version('rapidfuzz'))
print('huggingface_hub:', importlib.metadata.version('huggingface_hub'))


In [ ]:
import os
os.environ['HF_TOKEN'] = '<YOUR_HF_TOKEN>'
from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])

import numpy as np
import random
import torch

PIPELINE_VERSION = 'v7.5.5'
PROSPECT_SPEC_VERSION = 'v7.5.5'
PROMPT_FORMAT = 'format_c_qa_options_v7_5'

MODEL_ID = 'EleutherAI/gpt-j-6B'
MODEL_SHORT = 'gpt-j-6b-fp32'
MODEL_DTYPE = 'float32'

DRIVE_BASE = f'/content/drive/MyDrive/WCC/models/{MODEL_SHORT}'
PROSPECT_PHASE = '50_prospect'
PHASE_DIR = f'{DRIVE_BASE}/{PROSPECT_PHASE}'

WORD_PAIRS_PATH = f'{PHASE_DIR}/word_pairs_prospect.csv'
TRIALS_ALL_PATH = f'{PHASE_DIR}/trials_all_prospect.csv'
ACTIVATION_PATH = f'{PHASE_DIR}/activation_vectors_prospect.npz'
CLEAN_LOGITS_PATH = f'{PHASE_DIR}/clean_logits_prospect.npz'
ACTIVATION_CKPT_PATH = f'{PHASE_DIR}/activation_vectors_prospect_checkpoint.npz'
CLEAN_LOGITS_CKPT_PATH = f'{PHASE_DIR}/clean_logits_prospect_checkpoint.npz'
DOSE_RESPONSE_PATH = f'{PHASE_DIR}/prospect_dose_response_v2.csv'
PER_SEED_CELL_PATH = f'{PHASE_DIR}/per_seed_cell_classification_prospect.csv'
GENERATION_LOG_PATH = f'{PHASE_DIR}/generation_log_prospect.json'
ARCHITECTURE_PATH = f'{PHASE_DIR}/architecture_prospect.json'
CONFIG_PATH = f'{PHASE_DIR}/config_prospect.json'

N_PROSPECT_PAIRS = 100
PROSPECT_WORD_SEED = 2027
N_PROSPECT_SEEDS = 10
PROSPECT_SEEDS = list(range(43, 53))
CAL_FRAC = 0.20
RATIOS_PROSPECT = [0.01, 0.03, 0.05, 0.07, 0.10, 0.12, 0.15, 0.20, 0.30, 0.50]
ORDERINGS = [
    'hihp_imp_desc',
    'hihp_imp_asc',
    'hilp_imp_desc',
    'hilp_imp_asc',
    'C_imp_asc',
    'D_imp_asc',
    'diag_rank_asc', 'std_imp_asc',
]

PATCH_BATCH_SIZE = 128
PHASE0_COLLECT_BATCH = 32
PROGRESS_EVERY_BATCHES = 4
PARITY_CALIBRATION_N = 128
PARITY_QUANTILE_BINS = 4
PARITY_ATOL = 0.05
PARITY_HARD_CEILING = 0.10

PATCH_MECHANISM = 'input_overwrite_v5_1_pattern'
PATCH_MECHANISM_VERSION = 'v2_post_20260421_oooerror_fix'
RSA_METRIC = 'cosine_similarity_rdm'
CELL_NAMING = 'hihp_hilp_C_D'
DIAG_RANK_FORMULA = 'rank(rsa_max)+rank(perturbation_L2)'

ATTR_P = ('bold', 'cautious')
ATTR_Q = ('complex', 'simple')
ATTR_DIMS = ['P', 'Q']
CREL_PAIRS = [
    ('SAME', 'OPP'),
    ('SAME', 'MORE'),
    ('SAME', 'LESS'),
    ('OPP', 'MORE'),
    ('OPP', 'LESS'),
    ('MORE', 'LESS'),
]

TEMPLATE_SAMEOPP = """Definitions:
- {word_a}: something that is {attr1} and {attr2}
- {word_b} is {relation}

Q: Is {word_b} {choice1} or {choice2}? (Answer EXACTLY ONE WORD)
A:"""

TEMPLATE_COMPARE = """Definitions:
- {word_a}: something that is {attr1} and {attr2}
- {word_b} is {relation}

Q: Which is more {attr_asked}, {left_word} or {right_word}? (Answer EXACTLY ONE WORD)
A:"""

COMMON_WORD_LIST_SIZE = 3000
COMMON_WORD_LIST_URL = 'https://pypi.org/project/wordfreq/'
WORD_LENGTH_MIN = 8
WORD_LENGTH_MAX = 11
CANDIDATE_POOL_SIZE = 20_000
MAX_ATTEMPTS = 200_000
LEVENSHTEIN_MIN_DISTANCE = 3
FIRST_SUBTOKEN_MIN_LEN = 3
LOGIT_GAP_THRESHOLDS = (0.5, 1.0)
LOGIT_GAP_NEUTRAL_CONTEXT = 'A:'

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

assert len(PROSPECT_SEEDS) == N_PROSPECT_SEEDS

print(f'MODEL={MODEL_SHORT} VERSION={PIPELINE_VERSION} SPEC={PROSPECT_SPEC_VERSION}')
print(f'PHASE_DIR={PHASE_DIR}')
print(f'PATCH_BATCH_SIZE={PATCH_BATCH_SIZE} PHASE0_COLLECT_BATCH={PHASE0_COLLECT_BATCH}')


In [ ]:
import gc
import importlib.metadata
import json
import math
import os
import random
import time
from collections import defaultdict
from datetime import datetime

import nnsight
import numpy as np
import pandas as pd
import torch
import transformers
from nnsight import LanguageModel
from rapidfuzz.distance import Levenshtein as RFLevenshtein
from scipy.stats import rankdata
from wordfreq import top_n_list

assert torch.cuda.is_available(), 'GPU runtime required'

COMMON_WORD_LIST_SOURCE = (
    f"wordfreq.top_n_list('en', {COMMON_WORD_LIST_SIZE}, wordlist='large') "
    f"from wordfreq=={importlib.metadata.version('wordfreq')} ({COMMON_WORD_LIST_URL})"
)


def ts():
    return datetime.now().strftime('%Y-%m-%d %H:%M:%S')


def log(msg):
    print(f'[{ts()}] {msg}')


def ensure_dir(path):
    os.makedirs(path, exist_ok=True)
    return path


def get_v(obj):
    return getattr(obj, 'value', obj)


def to_jsonable(obj):
    if isinstance(obj, dict):
        return {k: to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, np.generic):
        return obj.item()
    if isinstance(obj, torch.Tensor):
        return obj.detach().cpu().tolist()
    return obj


def save_json(data, path, label):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w') as fout:
        json.dump(to_jsonable(data), fout, indent=2)
    log(f'Saved {label}: path={path}')


def save_df_csv(df, path, label):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    df = df if isinstance(df, pd.DataFrame) else pd.DataFrame(df)
    df.to_csv(path, index=False)
    log(f'Saved {label}: rows={len(df)} nan_count={int(df.isna().sum().sum()) if len(df.columns) else 0} path={path}')
    return df


def append_rows_csv(rows, path, label, columns):
    rows_df = pd.DataFrame(rows, columns=columns)
    if os.path.exists(path):
        prev = pd.read_csv(path)
        rows_df = pd.concat([prev, rows_df], ignore_index=True)
    return save_df_csv(rows_df[columns], path, label)


def save_npz(mapping, path, label):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    np.savez_compressed(path, **mapping)
    log(f'Saved {label}: entries={len(mapping)} path={path}')


def load_npz_dict(path, label):
    npz = np.load(path)
    out = {k: npz[k] for k in npz.files}
    npz.close()
    log(f'Loaded {label}: entries={len(out)} path={path}')
    return out


def pair_sort_key(value):
    text = str(value)
    if text.startswith('pair') and text[4:].isdigit():
        return int(text[4:])
    try:
        return int(text)
    except Exception:
        return text


def get_torch_dtype(dtype_name):
    if dtype_name == 'float32':
        return torch.float32
    if dtype_name == 'float16':
        return torch.float16
    return torch.bfloat16


def prepare_inputs(tokenizer, prompts, device='cuda'):
    tokenizer.padding_side = 'left'
    enc = tokenizer(list(prompts), return_tensors='pt', padding=True)
    input_ids = enc['input_ids'].to(device)
    attention_mask = enc['attention_mask'].to(device)
    position_ids = (attention_mask.cumsum(dim=-1) - 1).clamp(min=0)
    position_ids.masked_fill_(attention_mask == 0, 0)
    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'position_ids': position_ids,
    }


def load_model():
    log(f'Loading model {MODEL_ID} dtype={MODEL_DTYPE}')
    model = LanguageModel(MODEL_ID, dtype=torch.float32, device_map='auto')
    tokenizer = model.tokenizer
    if tokenizer.pad_token_id is None and getattr(tokenizer, 'eos_token_id', None) is not None:
        tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.padding_side = 'left'
    if hasattr(model, 'tokenizer') and model.tokenizer is not tokenizer:
        model.tokenizer.padding_side = 'left'
    cfg = getattr(model.config, 'text_config', model.config)
    arch = {
        'num_layers': int(getattr(cfg, 'num_hidden_layers', getattr(cfg, 'n_layer', 28))),
        'num_heads': int(cfg.num_attention_heads),
        'head_dim': int(getattr(cfg, 'head_dim', None) or (cfg.hidden_size // cfg.num_attention_heads)),
        'hidden_size': int(cfg.hidden_size),
        'vocab_size': int(cfg.vocab_size),
        'total_heads': int(getattr(cfg, 'num_hidden_layers', getattr(cfg, 'n_layer', 28)) * cfg.num_attention_heads),
        'torch_dtype': MODEL_DTYPE,
    }
    log(
        f'Loaded model: layers={arch["num_layers"]} heads={arch["num_heads"]} '
        f'head_dim={arch["head_dim"]} vocab={arch["vocab_size"]}'
    )
    return model, tokenizer, arch


def unload_model(model):
    try:
        del model
    except Exception:
        pass
    gc.collect()
    torch.cuda.empty_cache()
    log('Model unloaded, CUDA cache cleared.')


def first_subtoken_id(tokenizer, word):
    ids = tokenizer.encode(' ' + word, add_special_tokens=False)
    if not ids:
        raise ValueError(f'Empty encoding for {word!r}')
    return int(ids[0])


def compute_neutral_logits(model):
    with model.trace(LOGIT_GAP_NEUTRAL_CONTEXT):
        logits_proxy = model.output.logits[0, -1, :].save()
    logits = get_v(logits_proxy).detach().float().cpu()
    log(f"Cached neutral logits for {LOGIT_GAP_NEUTRAL_CONTEXT!r}: shape={tuple(logits.shape)}")
    return logits


def prior_logit_gap(neutral_logits, tokenizer, word_a, word_b):
    token_a = first_subtoken_id(tokenizer, word_a)
    token_b = first_subtoken_id(tokenizer, word_b)
    return abs(float(neutral_logits[token_a]) - float(neutral_logits[token_b]))


def load_common_words(n_words):
    raw = top_n_list('en', n_words, wordlist='large')
    return [w.lower() for w in raw if len(w) >= 2]


ONSETS = [
    'b', 'd', 'f', 'g', 'k', 'l', 'm', 'n', 'p', 'r', 's', 't', 'v', 'w', 'z',
    'br', 'fl', 'gr', 'kr', 'pl', 'pr', 'sl', 'sp', 'st', 'tr',
]
VOWELS = ['a', 'e', 'i', 'o', 'u', 'ai', 'ou', 'ei']
CODAS_MID = ['', 'n', 'l', 'r', 's']
CODAS_END = ['', 'n', 'm', 'l', 'r', 's', 't', 'k']


def gen_word(rng):
    syllables = []
    for idx in range(3):
        coda_pool = CODAS_MID if idx < 2 else CODAS_END
        syllables.append(rng.choice(ONSETS) + rng.choice(VOWELS) + rng.choice(coda_pool))
    return ''.join(syllables)


def _build_common_word_buckets(common_words):
    buckets = defaultdict(list)
    for word in common_words:
        buckets[len(word)].append(word)
    return dict(buckets)


def _far_enough_from_commons(word, buckets):
    lo = len(word) - (LEVENSHTEIN_MIN_DISTANCE - 1)
    hi = len(word) + (LEVENSHTEIN_MIN_DISTANCE - 1)
    for other_len in range(lo, hi + 1):
        for common_word in buckets.get(other_len, ()):
            distance = RFLevenshtein.distance(word, common_word, score_cutoff=LEVENSHTEIN_MIN_DISTANCE - 1)
            if distance < LEVENSHTEIN_MIN_DISTANCE:
                return False
    return True


def build_candidate_pool(common_words, seed, pool_size):
    rng = random.Random(seed)
    buckets = _build_common_word_buckets(common_words)
    pool = []
    seen = set()
    draws = 0
    max_draws = pool_size * 40
    while len(pool) < pool_size and draws < max_draws:
        draws += 1
        word = gen_word(rng).lower()
        if word in seen:
            continue
        if not (WORD_LENGTH_MIN <= len(word) <= WORD_LENGTH_MAX):
            continue
        if not _far_enough_from_commons(word, buckets):
            continue
        seen.add(word)
        pool.append(word)
    if len(pool) < pool_size:
        raise RuntimeError(f'Candidate pool underfilled: {len(pool)}/{pool_size} after {draws} draws')
    log(f'Candidate pool ready: {len(pool)} words after {draws} draws')
    return pool


def generate_word_pairs(tokenizer, neutral_logits, candidate_pool, threshold):
    rng = random.Random(PROSPECT_WORD_SEED + 17)
    accepted = []
    pair_gaps = []
    used = set()
    attempts = 0
    while len(accepted) < N_PROSPECT_PAIRS and attempts < MAX_ATTEMPTS:
        attempts += 1
        word_a, word_b = rng.sample(candidate_pool, 2)
        if word_a[0] == word_b[0]:
            continue
        key = tuple(sorted((word_a, word_b)))
        if key in used:
            continue
        ids_a = tokenizer.encode(' ' + word_a, add_special_tokens=False)
        ids_b = tokenizer.encode(' ' + word_b, add_special_tokens=False)
        if not ids_a or not ids_b:
            continue
        if ids_a[0] == ids_b[0]:
            continue
        if len(ids_a) != len(ids_b):
            continue
        sub_a = tokenizer.decode([ids_a[0]]).strip()
        sub_b = tokenizer.decode([ids_b[0]]).strip()
        if len(sub_a) < FIRST_SUBTOKEN_MIN_LEN or len(sub_b) < FIRST_SUBTOKEN_MIN_LEN:
            continue
        gap = prior_logit_gap(neutral_logits, tokenizer, word_a, word_b)
        if gap >= threshold:
            continue
        pair_id = f'pair{len(accepted)}'
        accepted.append({
            'pair_id': pair_id,
            'word_a': word_a,
            'word_b': word_b,
            'first_subtoken_id_a': int(ids_a[0]),
            'first_subtoken_id_b': int(ids_b[0]),
            'first_subtoken_text_a': sub_a,
            'first_subtoken_text_b': sub_b,
            'token_count_a': int(len(ids_a)),
            'token_count_b': int(len(ids_b)),
            'logit_gap_A_colon': float(gap),
        })
        pair_gaps.append({'pair_id': pair_id, 'word_a': word_a, 'word_b': word_b, 'gap': float(gap)})
        used.add(key)
    if len(accepted) < N_PROSPECT_PAIRS:
        return None
    return {
        'pairs': accepted,
        'pair_gaps': pair_gaps,
        'threshold_used': float(threshold),
        'attempts_used': int(attempts),
        'max_gap_observed': float(max(item['gap'] for item in pair_gaps)),
    }


def build_trials_all(word_pairs_df, tokenizer):
    p_pos, p_neg = ATTR_P
    q_pos, q_neg = ATTR_Q
    rows = []
    for row in word_pairs_df.to_dict('records'):
        pair_id = row['pair_id']
        word_a = row['word_a']
        word_b = row['word_b']

        for attr_dim, pos_word, neg_word in [('P', p_pos, p_neg), ('Q', q_pos, q_neg)]:
            rows.append({
                'trial_id': f'SAME_{attr_dim}_{pair_id}',
                'crel': 'SAME',
                'attr_dim': attr_dim,
                'pair_id': pair_id,
                'word_a': word_a,
                'word_b': word_b,
                'correct_answer': pos_word,
                'incorrect_answer': neg_word,
                'correct_first_subtoken_id': first_subtoken_id(tokenizer, pos_word),
                'incorrect_first_subtoken_id': first_subtoken_id(tokenizer, neg_word),
                'answer_type': 'attribute_word',
                'prompt': TEMPLATE_SAMEOPP.format(
                    word_a=word_a, word_b=word_b, attr1=p_pos, attr2=q_pos,
                    relation=f'the same as {word_a}', choice1=pos_word, choice2=neg_word,
                ),
            })
            rows.append({
                'trial_id': f'OPP_{attr_dim}_{pair_id}',
                'crel': 'OPP',
                'attr_dim': attr_dim,
                'pair_id': pair_id,
                'word_a': word_a,
                'word_b': word_b,
                'correct_answer': neg_word,
                'incorrect_answer': pos_word,
                'correct_first_subtoken_id': first_subtoken_id(tokenizer, neg_word),
                'incorrect_first_subtoken_id': first_subtoken_id(tokenizer, pos_word),
                'answer_type': 'attribute_word',
                'prompt': TEMPLATE_SAMEOPP.format(
                    word_a=word_a, word_b=word_b, attr1=p_pos, attr2=q_pos,
                    relation=f'the opposite of {word_a}', choice1=pos_word, choice2=neg_word,
                ),
            })

        for attr_dim, asked_word in [('P', p_pos), ('Q', q_pos)]:
            rows.append({
                'trial_id': f'MORE_{attr_dim}_{pair_id}',
                'crel': 'MORE',
                'attr_dim': attr_dim,
                'pair_id': pair_id,
                'word_a': word_a,
                'word_b': word_b,
                'correct_answer': word_b,
                'incorrect_answer': word_a,
                'correct_first_subtoken_id': first_subtoken_id(tokenizer, word_b),
                'incorrect_first_subtoken_id': first_subtoken_id(tokenizer, word_a),
                'answer_type': 'novel_word',
                'prompt': TEMPLATE_COMPARE.format(
                    word_a=word_a, word_b=word_b, attr1=p_pos, attr2=q_pos,
                    relation=f'more/greater than {word_a} in all respects',
                    attr_asked=asked_word, left_word=word_a, right_word=word_b,
                ),
            })
            rows.append({
                'trial_id': f'LESS_{attr_dim}_{pair_id}',
                'crel': 'LESS',
                'attr_dim': attr_dim,
                'pair_id': pair_id,
                'word_a': word_a,
                'word_b': word_b,
                'correct_answer': word_a,
                'incorrect_answer': word_b,
                'correct_first_subtoken_id': first_subtoken_id(tokenizer, word_a),
                'incorrect_first_subtoken_id': first_subtoken_id(tokenizer, word_b),
                'answer_type': 'novel_word',
                'prompt': TEMPLATE_COMPARE.format(
                    word_a=word_a, word_b=word_b, attr1=p_pos, attr2=q_pos,
                    relation=f'less/smaller than {word_a} in all respects',
                    attr_asked=asked_word, left_word=word_a, right_word=word_b,
                ),
            })
    return pd.DataFrame(rows)


@torch.no_grad()
def clean_forward(model, arch, prompt):
    acts_batch, logits_batch = clean_forward_batched(model, arch, [prompt])
    return acts_batch[0], logits_batch[0]


@torch.no_grad()
def clean_forward_batched(model, arch, prompts):
    inputs = prepare_inputs(model.tokenizer, prompts)
    head_inputs = []
    with model.trace(inputs):
        for layer_idx in range(arch['num_layers']):
            attn_in = model.transformer.h[layer_idx].attn.out_proj.input[:, -1, :]
            head_inputs.append(attn_in.view(len(prompts), arch['num_heads'], arch['head_dim']).save())
        logits_proxy = model.output.logits[:, -1, :].save()
    acts = torch.stack([get_v(x) for x in head_inputs], dim=1).detach().float().cpu().numpy()
    logits = get_v(logits_proxy).detach().float().cpu().numpy()
    return acts.astype(np.float32, copy=False), logits.astype(np.float32, copy=False)


@torch.no_grad()
def get_batched_group_logits(model, arch, prompts, source_vecs, patch_heads):
    if not patch_heads:
        raise ValueError('patch_heads must be non-empty')
    patches_by_layer = defaultdict(list)
    for layer_idx, head_idx in sorted(patch_heads, key=lambda x: (int(x[0]), int(x[1]))):
        patches_by_layer[int(layer_idx)].append(int(head_idx))
    src_stack = torch.as_tensor(np.stack(source_vecs), dtype=torch.float32, device='cuda')
    inputs = prepare_inputs(model.tokenizer, prompts)
    with model.trace(inputs):
        for layer_idx in sorted(patches_by_layer):
            proj = model.transformer.h[layer_idx].attn.out_proj
            for head_idx in patches_by_layer[layer_idx]:
                start = head_idx * arch['head_dim']
                end = (head_idx + 1) * arch['head_dim']
                proj.input[:, -1, start:end] = src_stack[:, layer_idx, head_idx]
        logits_proxy = model.output.logits[:, -1, :].save()
    return get_v(logits_proxy).detach().float().cpu().numpy().astype(np.float32, copy=False)


def ratio_to_k(ratio, total_heads):
    return max(1, int(math.ceil(float(ratio) * float(total_heads) - 1e-12)))


def prepare_ratio_grid(base_ratios, total_heads, max_equal_k_ratio):
    ratio_by_k = {}
    for raw_ratio in sorted(set([float(x) for x in list(base_ratios) + [max_equal_k_ratio]])):
        bounded = min(float(raw_ratio), 0.50)
        k = ratio_to_k(bounded, total_heads)
        if k not in ratio_by_k or bounded < ratio_by_k[k]:
            ratio_by_k[k] = bounded
    ratio_grid = [round(ratio_by_k[k], 6) for k in sorted(ratio_by_k)]
    return ratio_grid, round(min(float(max_equal_k_ratio), 0.50), 6)


def _batched_spearman_rows(matrix_2d, ref_vec):
    matrix_2d = np.asarray(matrix_2d, dtype=np.float64)
    ref_vec = np.asarray(ref_vec, dtype=np.float64)
    ref_mask = np.isfinite(ref_vec)
    if int(ref_mask.sum()) < 3:
        return np.full(matrix_2d.shape[0], np.nan, dtype=np.float64)
    mat = matrix_2d[:, ref_mask]
    ref = ref_vec[ref_mask]
    valid_rows = np.isfinite(mat).all(axis=1)
    out = np.full(mat.shape[0], np.nan, dtype=np.float64)
    if not valid_rows.any():
        return out
    mat_valid = mat[valid_rows]
    rank_mat = rankdata(mat_valid, axis=1)
    rank_ref = rankdata(ref)
    rank_mat = rank_mat - rank_mat.mean(axis=1, keepdims=True)
    rank_ref = rank_ref - rank_ref.mean()
    denom = np.sqrt((rank_mat ** 2).sum(axis=1) * (rank_ref ** 2).sum())
    good = denom > 0
    vals = np.full(mat_valid.shape[0], np.nan, dtype=np.float64)
    vals[good] = (rank_mat[good] @ rank_ref) / denom[good]
    out[np.where(valid_rows)[0]] = vals
    return out


def compute_rsa_and_perturbation(all_vecs, trials_df, cal_pair_ids, arch):
    cal_df = trials_df[trials_df['pair_id'].isin(set(cal_pair_ids))].copy()
    cal_df = cal_df.sort_values(['pair_id', 'crel', 'attr_dim'], kind='mergesort').reset_index(drop=True)
    trial_ids = cal_df['trial_id'].tolist()
    missing = [tid for tid in trial_ids if tid not in all_vecs]
    if missing:
        raise KeyError(f'Missing activation vectors for calibration trials: {missing[:5]}')

    cal_matrix = np.stack([all_vecs[tid] for tid in trial_ids]).astype(np.float32, copy=False)
    t, layers, heads, dim = cal_matrix.shape
    n_heads = layers * heads

    head_matrix = cal_matrix.transpose(1, 2, 0, 3).reshape(n_heads, t, dim)
    norms = np.linalg.norm(head_matrix, axis=2, keepdims=True)
    valid_rows = np.isfinite(head_matrix).all(axis=(1, 2)) & np.isfinite(norms).all(axis=(1, 2)) & (norms[:, 0, 0] > 1e-12)
    normed = np.divide(head_matrix, norms, out=np.full_like(head_matrix, np.nan), where=norms > 1e-12)
    sim = np.einsum('htd,hsd->hts', normed, normed, optimize=True)
    tri_i, tri_j = np.triu_indices(t, k=1)
    neural_upper = sim[:, tri_i, tri_j].astype(np.float64, copy=False)
    neural_upper[~valid_rows] = np.nan

    crel_codes = pd.Categorical(cal_df['crel']).codes
    attr_codes = pd.Categorical(cal_df['attr_dim']).codes
    crel_ref = (crel_codes[:, None] == crel_codes[None, :])[tri_i, tri_j].astype(np.float64)
    attr_ref = (attr_codes[:, None] == attr_codes[None, :])[tri_i, tri_j].astype(np.float64)

    rsa_crel = _batched_spearman_rows(neural_upper, crel_ref)
    rsa_attr = _batched_spearman_rows(neural_upper, attr_ref)
    pairwise = np.stack([rsa_crel, rsa_attr], axis=1)
    has_any = np.isfinite(pairwise).any(axis=1)
    rsa_before = np.full(n_heads, np.nan, dtype=np.float64)
    rsa_before[has_any] = np.nanmax(pairwise[has_any], axis=1)
    rsa_max = np.where(np.isfinite(rsa_before), np.maximum(rsa_before, 0.0), np.nan)

    tid_to_idx = {tid: idx for idx, tid in enumerate(trial_ids)}
    pert_pairs = []
    for src_crel, tgt_crel in CREL_PAIRS:
        for pair_id in sorted(cal_df['pair_id'].unique().tolist(), key=pair_sort_key):
            for attr_dim in ATTR_DIMS:
                src_tid = f'{src_crel}_{attr_dim}_{pair_id}'
                tgt_tid = f'{tgt_crel}_{attr_dim}_{pair_id}'
                if src_tid in tid_to_idx and tgt_tid in tid_to_idx:
                    pert_pairs.append((tid_to_idx[src_tid], tid_to_idx[tgt_tid]))
    if pert_pairs:
        src_idx = np.array([x[0] for x in pert_pairs], dtype=np.int64)
        tgt_idx = np.array([x[1] for x in pert_pairs], dtype=np.int64)
        diffs = head_matrix[:, src_idx, :] - head_matrix[:, tgt_idx, :]
        dists = np.linalg.norm(diffs.astype(np.float32, copy=False), axis=2)
        pert = np.nanmean(dists, axis=1).astype(np.float64, copy=False)
    else:
        pert = np.full(n_heads, np.nan, dtype=np.float64)

    layer_ids = np.repeat(np.arange(layers), heads)
    head_ids = np.tile(np.arange(heads), layers)
    rsa_df = pd.DataFrame({
        'layer': layer_ids.astype(int),
        'head': head_ids.astype(int),
        'rsa_crel': rsa_crel,
        'rsa_attr': rsa_attr,
        'rsa_max_before_clamp': rsa_before,
        'rsa_max': rsa_max,
    })
    pert_df = pd.DataFrame({
        'layer': layer_ids.astype(int),
        'head': head_ids.astype(int),
        'perturbation_L2': pert,
    })
    return rsa_df, pert_df


def classify_cells(rsa_df, pert_df):
    merged = rsa_df.merge(pert_df, on=['layer', 'head'], how='outer').copy()
    valid = np.isfinite(merged['rsa_max'].to_numpy(dtype=np.float64)) & np.isfinite(
        merged['perturbation_L2'].to_numpy(dtype=np.float64)
    )
    if not valid.any():
        raise RuntimeError('No valid heads available for cell classification')
    rsa_median = float(np.nanmedian(merged.loc[valid, 'rsa_max'].to_numpy(dtype=np.float64)))
    pert_median = float(np.nanmedian(merged.loc[valid, 'perturbation_L2'].to_numpy(dtype=np.float64)))
    cells = []
    for _, row in merged.iterrows():
        rsa_value = float(row['rsa_max']) if pd.notna(row['rsa_max']) else np.nan
        pert_value = float(row['perturbation_L2']) if pd.notna(row['perturbation_L2']) else np.nan
        if not (np.isfinite(rsa_value) and np.isfinite(pert_value)):
            cells.append(np.nan)
            continue
        hi_r = rsa_value >= rsa_median
        hi_p = pert_value >= pert_median
        if hi_r and hi_p:
            cells.append('hihp')
        elif hi_r and not hi_p:
            cells.append('hilp')
        elif (not hi_r) and hi_p:
            cells.append('C')
        else:
            cells.append('D')
    merged['cell'] = cells
    merged['rsa_median'] = rsa_median
    merged['perturbation_median'] = pert_median
    return merged, {
        'rsa_median': rsa_median,
        'perturbation_median': pert_median,
        'excluded_nan_heads': int((~valid).sum()),
    }


def build_orderings(cell_df):
    """7 orderings per factual_recall pattern."""
    orderings = {}
    for cell_letter in ('hihp', 'hilp'):
        sub = cell_df[cell_df['cell'] == cell_letter].dropna(subset=['rsa_max'])
        desc = sub.sort_values(['rsa_max', 'layer', 'head'], ascending=[False, True, True])
        asc = sub.sort_values(['rsa_max', 'layer', 'head'], ascending=[True, True, True])
        orderings[f'{cell_letter}_imp_desc'] = list(zip(desc['layer'].astype(int), desc['head'].astype(int)))
        orderings[f'{cell_letter}_imp_asc'] = list(zip(asc['layer'].astype(int), asc['head'].astype(int)))
    for cell_letter in ('C', 'D'):
        sub = cell_df[cell_df['cell'] == cell_letter].dropna(subset=['rsa_max'])
        asc = sub.sort_values(['rsa_max', 'layer', 'head'], ascending=[True, True, True])
        orderings[f'{cell_letter}_imp_asc'] = list(zip(asc['layer'].astype(int), asc['head'].astype(int)))
    # diag_rank_asc: C∪D pool only (NOT full pool), joint rank (imp + pert) ascending
    low_imp = cell_df[cell_df['cell'].isin(['C', 'D'])].dropna(subset=['rsa_max', 'perturbation_L2']).reset_index(drop=True)
    if len(low_imp) > 0:
        imp_arr = low_imp['rsa_max'].to_numpy()
        pert_arr = low_imp['perturbation_L2'].to_numpy()
        joint_rank = rankdata(imp_arr) + rankdata(pert_arr)
        ordered = low_imp.iloc[joint_rank.argsort()]
        orderings['diag_rank_asc'] = list(zip(ordered['layer'].astype(int), ordered['head'].astype(int)))
    else:
        orderings['diag_rank_asc'] = []
    # std_imp_asc: full candidate pool, imp ascending (pooled importance-only baseline; Main pipeline schema)
    sub_all = cell_df.dropna(subset=['rsa_max']).sort_values(['rsa_max', 'layer', 'head'], ascending=[True, True, True])
    orderings['std_imp_asc'] = list(zip(sub_all['layer'].astype(int), sub_all['head'].astype(int)))
    return orderings


def build_eval_patch_trials(trials_df, eval_pair_ids):
    eval_set = set(eval_pair_ids)
    trial_by_key = {(row['crel'], row['pair_id'], row['attr_dim']): row for row in trials_df.to_dict('records')}
    patch_trials = []
    for src_crel, tgt_crel in CREL_PAIRS:
        for pair_id in sorted(eval_set, key=pair_sort_key):
            for attr_dim in ATTR_DIMS:
                src_row = trial_by_key[(src_crel, pair_id, attr_dim)]
                tgt_row = trial_by_key[(tgt_crel, pair_id, attr_dim)]
                patch_trials.append({
                    'src_tid': src_row['trial_id'],
                    'tgt_tid': tgt_row['trial_id'],
                    'prompt': tgt_row['prompt'],
                    'pair_id': pair_id,
                    'attr_dim': attr_dim,
                    'crel_pair': f'{src_crel}-{tgt_crel}',
                    'tgt_tok': int(tgt_row['correct_first_subtoken_id']),
                })
    return patch_trials


def run_group_ratio_eval(model, arch, patch_trials, all_vecs, clean_logits_dict, active_heads):
    if not active_heads:
        return [float('nan')] * len(patch_trials)
    deltas = []
    for batch_start in range(0, len(patch_trials), PATCH_BATCH_SIZE):
        batch = patch_trials[batch_start:batch_start + PATCH_BATCH_SIZE]
        prompts = [row['prompt'] for row in batch]
        source_vecs = [all_vecs[row['src_tid']] for row in batch]
        logits_batch = get_batched_group_logits(model, arch, prompts, source_vecs, active_heads)
        for idx, row in enumerate(batch):
            clean = clean_logits_dict[row['tgt_tid']]
            delta = float(logits_batch[idx, row['tgt_tok']] - clean[row['tgt_tok']])
            deltas.append(abs(delta))
    return deltas


def parity_check_batched_vs_solo(model, arch, trials_df, clean_logits):
    """Non-halting parity audit: returns stats dict. Called AFTER main loop for post-hoc inspection.
    Semantic checks (argmax, D+ sign) and drift (observed_max) are logged but do NOT raise.
    """
    tokenizer = model.tokenizer
    work = trials_df.copy()
    work['token_len'] = work['prompt'].map(lambda p: len(tokenizer.encode(p)))
    work = work.sort_values(['token_len', 'trial_id'], kind='mergesort').reset_index(drop=True)
    bins = np.array_split(work.index.to_numpy(), PARITY_QUANTILE_BINS)
    per_bin = PARITY_CALIBRATION_N // PARITY_QUANTILE_BINS
    chosen = []
    for bin_idx, bin_ids in enumerate(bins):
        if len(bin_ids) < per_bin:
            log(f'WARNING parity bin {bin_idx} underfilled: {len(bin_ids)} < {per_bin} (use all)')
            chosen.extend(bin_ids.tolist())
        else:
            chosen.extend(bin_ids[:per_bin].tolist())
    sample = work.loc[chosen].reset_index(drop=True)
    prompts = sample['prompt'].tolist()
    batched_logits = clean_forward_batched(model, arch, prompts)[1]
    max_diffs, sign_matches, argmax_matches = [], [], []
    for idx, row in sample.iterrows():
        solo = np.asarray(clean_logits[row['trial_id']], dtype=np.float32)
        batched = np.asarray(batched_logits[idx], dtype=np.float32)
        max_diffs.append(float(np.abs(batched - solo).max()))
        argmax_matches.append(int(np.argmax(batched)) == int(np.argmax(solo)))
        solo_d = float(solo[int(row['correct_first_subtoken_id'])] - solo[int(row['incorrect_first_subtoken_id'])])
        batch_d = float(batched[int(row['correct_first_subtoken_id'])] - batched[int(row['incorrect_first_subtoken_id'])])
        sign_matches.append((solo_d == 0.0 and batch_d == 0.0) or (solo_d * batch_d > 0.0))
    observed_max = float(max(max_diffs)) if max_diffs else float('nan')
    argmax_match_rate = f'{sum(argmax_matches)}/{len(argmax_matches)}'
    sign_match_rate = f'{sum(sign_matches)}/{len(sign_matches)}'
    if not all(argmax_matches):
        log(f'WARNING parity argmax mismatch: {argmax_match_rate} (investigate post-run)')
    if not all(sign_matches):
        log(f'WARNING parity D+ sign mismatch: {sign_match_rate} (investigate post-run)')
    if observed_max > PARITY_HARD_CEILING:
        log(f'WARNING parity observed_max={observed_max:.4f} > ceiling {PARITY_HARD_CEILING} (drift info, not halt)')
    stats = {
        'n_trials': int(len(sample)),
        'quantile_bins': int(PARITY_QUANTILE_BINS),
        'per_bin': int(per_bin),
        'token_len_min': int(sample['token_len'].min()) if len(sample) else None,
        'token_len_max': int(sample['token_len'].max()) if len(sample) else None,
        'observed_max_diff': observed_max,
        'argmax_match_rate': argmax_match_rate,
        'dplus_sign_match_rate': sign_match_rate,
        'atol': float(PARITY_ATOL),
        'hard_ceiling': float(PARITY_HARD_CEILING),
    }
    log(f'Parity audit: N={stats["n_trials"]} argmax={argmax_match_rate} D+sign={sign_match_rate} max_diff={observed_max:.4f}')
    return stats



In [ ]:
# Phase 0' — Drive-direct word pair generation
phase0_word_t0 = time.time()

if os.path.exists(WORD_PAIRS_PATH) and os.path.exists(TRIALS_ALL_PATH):
    word_pairs_df = pd.read_csv(WORD_PAIRS_PATH)
    trials_all_df = pd.read_csv(TRIALS_ALL_PATH)
    log(f"Phase 0' cached: word_pairs={len(word_pairs_df)} trials_all={len(trials_all_df)}")
else:
    model, tokenizer, arch = load_model()
    save_json(arch, ARCHITECTURE_PATH, 'architecture_prospect.json')

    common_words = load_common_words(COMMON_WORD_LIST_SIZE)
    candidate_pool = build_candidate_pool(common_words, PROSPECT_WORD_SEED, CANDIDATE_POOL_SIZE)
    neutral_logits = compute_neutral_logits(model)

    generation_result = None
    for threshold in LOGIT_GAP_THRESHOLDS:
        log(f'Trying word-pair generation with threshold < {threshold}')
        generation_result = generate_word_pairs(tokenizer, neutral_logits, candidate_pool, threshold)
        if generation_result is not None:
            break
    if generation_result is None:
        raise RuntimeError('Prospect word-pair generation failed under all thresholds')

    word_pairs_df = save_df_csv(
        pd.DataFrame(generation_result['pairs']),
        WORD_PAIRS_PATH,
        'word_pairs_prospect.csv',
    )
    trials_all_df = save_df_csv(
        build_trials_all(word_pairs_df, tokenizer),
        TRIALS_ALL_PATH,
        'trials_all_prospect.csv',
    )
    save_json(
        {
            'threshold_used': generation_result['threshold_used'],
            'attempts_used': generation_result['attempts_used'],
            'max_gap_observed': generation_result['max_gap_observed'],
            'candidate_pool_size': CANDIDATE_POOL_SIZE,
            'common_word_list_source': COMMON_WORD_LIST_SOURCE,
            'pair_gaps': generation_result['pair_gaps'],
        },
        GENERATION_LOG_PATH,
        'generation_log_prospect.json',
    )
    log(f"Phase 0' complete in {time.time() - phase0_word_t0:.1f}s")

assert len(word_pairs_df) == N_PROSPECT_PAIRS
assert len(trials_all_df) == 4 * 2 * N_PROSPECT_PAIRS


In [ ]:
# Phase 0'' — batched activation + clean-logit collection
phase0_collect_t0 = time.time()

if os.path.exists(ACTIVATION_PATH) and os.path.exists(CLEAN_LOGITS_PATH):
    activation_vectors = load_npz_dict(ACTIVATION_PATH, 'activation_vectors_prospect.npz')
    clean_logits = load_npz_dict(CLEAN_LOGITS_PATH, 'clean_logits_prospect.npz')
    if 'arch' not in globals() or arch is None:
        with open(ARCHITECTURE_PATH) as fin:
            arch = json.load(fin)
    log(f"Phase 0'' cached: activations={len(activation_vectors)} clean_logits={len(clean_logits)}")
else:
    if 'model' not in globals() or model is None:
        model, tokenizer, arch = load_model()
    elif 'tokenizer' not in globals() or tokenizer is None:
        tokenizer = model.tokenizer
    save_json(arch, ARCHITECTURE_PATH, 'architecture_prospect.json')

    activation_vectors = load_npz_dict(ACTIVATION_CKPT_PATH, 'activation_vectors_prospect_checkpoint.npz') if os.path.exists(ACTIVATION_CKPT_PATH) else {}
    clean_logits = load_npz_dict(CLEAN_LOGITS_CKPT_PATH, 'clean_logits_prospect_checkpoint.npz') if os.path.exists(CLEAN_LOGITS_CKPT_PATH) else {}

    trial_rows = trials_all_df.sort_values(['pair_id', 'crel', 'attr_dim'], kind='mergesort').to_dict('records')
    remaining = [row for row in trial_rows if row['trial_id'] not in activation_vectors or row['trial_id'] not in clean_logits]
    total = len(trial_rows)
    log(f"Phase 0'' remaining={len(remaining)} / total={total}")

    for batch_idx, batch_start in enumerate(range(0, len(remaining), PHASE0_COLLECT_BATCH), start=1):
        batch = remaining[batch_start:batch_start + PHASE0_COLLECT_BATCH]
        prompts = [row['prompt'] for row in batch]
        acts_batch, logits_batch = clean_forward_batched(model, arch, prompts)
        for i, row in enumerate(batch):
            trial_id = row['trial_id']
            activation_vectors[trial_id] = acts_batch[i].astype(np.float32, copy=False)
            clean_logits[trial_id] = logits_batch[i].astype(np.float32, copy=False)
        save_npz(activation_vectors, ACTIVATION_CKPT_PATH, 'activation_vectors_prospect_checkpoint.npz')
        save_npz(clean_logits, CLEAN_LOGITS_CKPT_PATH, 'clean_logits_prospect_checkpoint.npz')
        if batch_idx % PROGRESS_EVERY_BATCHES == 0 or batch_start + len(batch) >= len(remaining):
            done = len(activation_vectors)
            elapsed = max(time.time() - phase0_collect_t0, 1e-6)
            rate = done / elapsed
            log(f"Phase 0'' progress: {done}/{total} trials rate={rate:.2f}/s")

    save_npz(activation_vectors, ACTIVATION_PATH, 'activation_vectors_prospect.npz')
    save_npz(clean_logits, CLEAN_LOGITS_PATH, 'clean_logits_prospect.npz')
    if os.path.exists(ACTIVATION_CKPT_PATH):
        os.remove(ACTIVATION_CKPT_PATH)
    if os.path.exists(CLEAN_LOGITS_CKPT_PATH):
        os.remove(CLEAN_LOGITS_CKPT_PATH)
    log(f"Phase 0'' complete in {time.time() - phase0_collect_t0:.1f}s")


In [ ]:
# Phase 1' setup — load artefacts and build remaining work
phase1_setup_t0 = time.time()

word_pairs_df = pd.read_csv(WORD_PAIRS_PATH)
trials_all_df = pd.read_csv(TRIALS_ALL_PATH)
all_vecs = load_npz_dict(ACTIVATION_PATH, 'activation_vectors_prospect.npz')
clean_logits = load_npz_dict(CLEAN_LOGITS_PATH, 'clean_logits_prospect.npz')

if os.path.exists(ARCHITECTURE_PATH):
    with open(ARCHITECTURE_PATH) as fin:
        arch = json.load(fin)
else:
    sample_shape = next(iter(all_vecs.values())).shape
    arch = {
        'num_layers': int(sample_shape[0]),
        'num_heads': int(sample_shape[1]),
        'head_dim': int(sample_shape[2]),
        'hidden_size': int(sample_shape[1] * sample_shape[2]),
        'total_heads': int(sample_shape[0] * sample_shape[1]),
        'torch_dtype': MODEL_DTYPE,
    }

dose_columns = [
    'model', 'seed', 'group', 'ratio', 'k', 'k_actual',
    'mean_abs_delta', 'median_abs_delta',
    'n_eval_trials', 'n_cal_pairs', 'n_eval_pairs',
    'n_cell_hihp', 'n_cell_hilp', 'n_cell_C', 'n_cell_D',
]
cell_columns = [
    'seed', 'layer', 'head', 'cell',
    'rsa_max', 'rsa_max_before_clamp', 'perturbation_L2', 'cal_pair_ids',
]

existing_dose_df = pd.read_csv(DOSE_RESPONSE_PATH) if os.path.exists(DOSE_RESPONSE_PATH) else pd.DataFrame(columns=dose_columns)
per_seed_cell_df = pd.read_csv(PER_SEED_CELL_PATH) if os.path.exists(PER_SEED_CELL_PATH) else pd.DataFrame(columns=cell_columns)

existing_dose_df = existing_dose_df[dose_columns] if len(existing_dose_df) else pd.DataFrame(columns=dose_columns)
per_seed_cell_df = per_seed_cell_df[cell_columns] if len(per_seed_cell_df) else pd.DataFrame(columns=cell_columns)

TOTAL_HEADS = int(arch['total_heads'])
completed_combos = set()
if len(existing_dose_df):
    for row in existing_dose_df[['seed', 'group', 'ratio']].drop_duplicates().to_dict('records'):
        completed_combos.add((int(row['seed']), str(row['group']), round(float(row['ratio']), 6)))

remaining_seed_orderings = []
for seed in PROSPECT_SEEDS:
    seed_done = existing_dose_df[existing_dose_df['seed'] == seed] if len(existing_dose_df) else pd.DataFrame(columns=dose_columns)
    done_groups = set(seed_done['group'].astype(str).unique().tolist()) if len(seed_done) else set()
    missing_groups = [group for group in ORDERINGS if group not in done_groups]
    if missing_groups:
        remaining_seed_orderings.append({'seed': int(seed), 'missing_groups': missing_groups})

log(
    f"Phase 1 setup: pairs={len(word_pairs_df)} trials={len(trials_all_df)} "
    f"completed_combos={len(completed_combos)} remaining_seed_orderings={len(remaining_seed_orderings)}"
)
phase1_setup_elapsed_sec = time.time() - phase1_setup_t0


In [ ]:
# Phase 1' main loop — parity, per-seed classification, dose-response append
phase1_main_t0 = time.time()

if 'model' not in globals() or model is None:
    model, tokenizer, arch_live = load_model()
else:
    arch_live = arch
if 'tokenizer' not in globals() or tokenizer is None:
    tokenizer = model.tokenizer
prospect_dose_response_df = existing_dose_df.copy()
per_seed_cell_df_live = per_seed_cell_df.copy()
pair_ids = sorted(word_pairs_df['pair_id'].tolist(), key=pair_sort_key)

for seed in PROSPECT_SEEDS:
    rng = random.Random(seed)
    shuffled = list(pair_ids)
    rng.shuffle(shuffled)
    n_cal_pairs = int(len(shuffled) * CAL_FRAC)
    cal_pairs = sorted(shuffled[:n_cal_pairs], key=pair_sort_key)
    eval_pairs = sorted(shuffled[n_cal_pairs:], key=pair_sort_key)

    rsa_df, pert_df = compute_rsa_and_perturbation(all_vecs, trials_all_df, cal_pairs, arch)
    cell_df, thresholds = classify_cells(rsa_df, pert_df)
    orderings_map = build_orderings(cell_df)
    cell_counts = {
        'hihp': int((cell_df['cell'] == 'hihp').sum()),
        'hilp': int((cell_df['cell'] == 'hilp').sum()),
        'C': int((cell_df['cell'] == 'C').sum()),
        'D': int((cell_df['cell'] == 'D').sum()),
    }

    max_equal_k_ratio = (
        min(cell_counts.get('C', 0), cell_counts.get('D', 0)) / float(TOTAL_HEADS)
        if TOTAL_HEADS > 0 else 0.0
    )
    ratio_grid, max_equal_k_ratio_applied = prepare_ratio_grid(RATIOS_PROSPECT, TOTAL_HEADS, max_equal_k_ratio)

    per_seed_rows = cell_df[['layer', 'head', 'cell', 'rsa_max', 'rsa_max_before_clamp', 'perturbation_L2']].copy()
    per_seed_rows.insert(0, 'seed', int(seed))
    per_seed_rows['cal_pair_ids'] = ','.join(cal_pairs)
    per_seed_cell_df_live = per_seed_cell_df_live[per_seed_cell_df_live['seed'] != seed].copy()
    per_seed_cell_df_live = pd.concat([per_seed_cell_df_live, per_seed_rows], ignore_index=True)
    per_seed_cell_df_live = save_df_csv(per_seed_cell_df_live[cell_columns], PER_SEED_CELL_PATH, 'per_seed_cell_classification_prospect.csv')

    eval_patch_trials = build_eval_patch_trials(trials_all_df, eval_pairs)

    for group_name in ORDERINGS:
        for ratio in ratio_grid:
            combo_key = (int(seed), str(group_name), round(float(ratio), 6))
            if combo_key in completed_combos:
                continue

            ordered_heads = orderings_map[group_name]
            k = ratio_to_k(ratio, TOTAL_HEADS)
            k_actual = min(k, len(ordered_heads))
            active_heads = ordered_heads[:k_actual]
            deltas = run_group_ratio_eval(model, arch_live, eval_patch_trials, all_vecs, clean_logits, active_heads)
            deltas_arr = np.asarray(deltas, dtype=np.float64)

            row = {
                'model': MODEL_SHORT,
                'seed': int(seed),
                'group': group_name,
                'ratio': float(ratio),
                'k': int(k),
                'k_actual': int(k_actual),
                'mean_abs_delta': float(np.nanmean(deltas_arr)) if np.isfinite(deltas_arr).any() else np.nan,
                'median_abs_delta': float(np.nanmedian(deltas_arr)) if np.isfinite(deltas_arr).any() else np.nan,
                'n_eval_trials': int(len(eval_patch_trials)),
                'n_cal_pairs': int(len(cal_pairs)),
                'n_eval_pairs': int(len(eval_pairs)),
                'n_cell_hihp': int(cell_counts.get('hihp', 0)),
                'n_cell_hilp': int(cell_counts.get('hilp', 0)),
                'n_cell_C': int(cell_counts.get('C', 0)),
                'n_cell_D': int(cell_counts.get('D', 0)),
            }
            prospect_dose_response_df = pd.concat([prospect_dose_response_df, pd.DataFrame([row])], ignore_index=True)
            prospect_dose_response_df = save_df_csv(
                prospect_dose_response_df[dose_columns],
                DOSE_RESPONSE_PATH,
                'prospect_dose_response_v2.csv',
            )
            completed_combos.add(combo_key)
            log(
                f"Phase 1 combo done: seed={seed} group={group_name} ratio={ratio:.6f} "
                f"k={k} k_actual={k_actual} max_equal_k_ratio_applied={max_equal_k_ratio_applied:.6f}"
            )

# Post-loop parity audit (non-halting, saved to config for post-hoc inspection)

try:

    parity_stats = parity_check_batched_vs_solo(model, arch_live, trials_all_df, clean_logits)

except Exception as _e:

    log(f"WARNING parity audit failed: {type(_e).__name__}: {_e}")

    parity_stats = {"error": type(_e).__name__, "msg": str(_e)}


phase50_elapsed_sec = time.time() - phase1_main_t0


In [ ]:
# Summary — gap_cd and gap_hihp_hilp
if not os.path.exists(DOSE_RESPONSE_PATH):
    raise FileNotFoundError(DOSE_RESPONSE_PATH)

prospect_dose_response_df = pd.read_csv(DOSE_RESPONSE_PATH)
if len(prospect_dose_response_df) == 0:
    log('Summary skipped: prospect_dose_response_v2.csv is empty')
else:
    pivot = prospect_dose_response_df.pivot_table(
        index=['seed', 'ratio'],
        columns='group',
        values='mean_abs_delta',
        aggfunc='first',
    ).reset_index()

    summary_rows = []
    for ratio, ratio_df in pivot.groupby('ratio', sort=True):
        if {'C_imp_asc', 'D_imp_asc'}.issubset(ratio_df.columns):
            gap_cd = ratio_df['C_imp_asc'].abs() - ratio_df['D_imp_asc'].abs()
            finite = pd.to_numeric(gap_cd, errors='coerce').dropna()
            summary_rows.append({
                'ratio': float(ratio),
                'metric': 'gap_cd',
                'mean': float(finite.mean()) if len(finite) else np.nan,
                'se': float(finite.std(ddof=1) / np.sqrt(len(finite))) if len(finite) > 1 else np.nan,
                'sign_consistency': f"{int((finite > 0).sum())}/{int(len(finite))}",
            })
        if {'hihp_imp_desc', 'hilp_imp_desc'}.issubset(ratio_df.columns):
            gap_hl = ratio_df['hihp_imp_desc'].abs() - ratio_df['hilp_imp_desc'].abs()
            finite = pd.to_numeric(gap_hl, errors='coerce').dropna()
            summary_rows.append({
                'ratio': float(ratio),
                'metric': 'gap_hihp_hilp',
                'mean': float(finite.mean()) if len(finite) else np.nan,
                'se': float(finite.std(ddof=1) / np.sqrt(len(finite))) if len(finite) > 1 else np.nan,
                'sign_consistency': f"{int((finite > 0).sum())}/{int(len(finite))}",
            })

    summary_df = pd.DataFrame(summary_rows)
    if len(summary_df):
        print(summary_df.sort_values(['ratio', 'metric']).to_string(index=False))
    else:
        log('Summary table empty')


In [ ]:
# Final artefact check + config save + runtime release
final_files = [
    WORD_PAIRS_PATH,
    TRIALS_ALL_PATH,
    ACTIVATION_PATH,
    CLEAN_LOGITS_PATH,
    DOSE_RESPONSE_PATH,
    PER_SEED_CELL_PATH,
    GENERATION_LOG_PATH,
    ARCHITECTURE_PATH,
]
missing = [path for path in final_files if not os.path.exists(path)]
if missing:
    raise FileNotFoundError(f'Missing artefacts: {missing}')

config_payload = {
    'pipeline_version': PIPELINE_VERSION,
    'prospect_spec_version': PROSPECT_SPEC_VERSION,
    'prompt_format': PROMPT_FORMAT,
    'model_id': MODEL_ID,
    'model_short': MODEL_SHORT,
    'model_dtype': MODEL_DTYPE,
    'phase_dir': PHASE_DIR,
    'n_prospect_pairs': N_PROSPECT_PAIRS,
    'prospect_word_seed': PROSPECT_WORD_SEED,
    'prospect_seeds': PROSPECT_SEEDS,
    'cal_frac': CAL_FRAC,
    'ratios_prospect': RATIOS_PROSPECT,
    'orderings': ORDERINGS,
    'patch_batch_size': PATCH_BATCH_SIZE,
    'phase0_collect_batch': PHASE0_COLLECT_BATCH,
    'parity_calibration_n': PARITY_CALIBRATION_N,
    'patch_mechanism': PATCH_MECHANISM,
    'patch_mechanism_version': PATCH_MECHANISM_VERSION,
    'rsa_metric': RSA_METRIC,
    'cell_naming': CELL_NAMING,
    'diag_rank_formula': DIAG_RANK_FORMULA,
    'common_word_list_source': COMMON_WORD_LIST_SOURCE,
    'n_word_pairs': int(len(pd.read_csv(WORD_PAIRS_PATH))),
    'n_trials_all': int(len(pd.read_csv(TRIALS_ALL_PATH))),
    'n_activation_entries': int(len(np.load(ACTIVATION_PATH).files)),
    'n_clean_logit_entries': int(len(np.load(CLEAN_LOGITS_PATH).files)),
    'n_dose_rows': int(len(pd.read_csv(DOSE_RESPONSE_PATH))),
    'n_per_seed_cell_rows': int(len(pd.read_csv(PER_SEED_CELL_PATH))),
    'phase1_setup_elapsed_sec': round(float(globals().get('phase1_setup_elapsed_sec', 0.0)), 3),
    'phase50_elapsed_sec': round(float(globals().get('phase50_elapsed_sec', 0.0)), 3),
    'parity_stats': globals().get('parity_stats', {}),
    'saved_at': datetime.now().isoformat(),
}
save_json(config_payload, CONFIG_PATH, 'config_prospect.json')

from google.colab import runtime
log('Releasing Colab runtime.')
runtime.unassign()
